# Proyecto Final: Generación de Criaturas Sintéticas (Pokémon) mediante Redes GAN
**Autor:** jorgeress  
**Fecha de entrega:** 5 de enero de 2026


## 1. Descripción del Problema
El objetivo principal es el desarrollo de un modelo de **Deep Learning** capaz de realizar síntesis de imágenes. En lugar de clasificar imágenes existentes, buscamos generar contenido nuevo.

### El Desafío
Entrenar una **GAN (Generative Adversarial Network)** presenta retos de equilibrio dinámico. El modelo debe aprender a mapear un espacio latente de 100 dimensiones (ruido aleatorio) a una distribución de píxeles que el ojo humano identifique como un "Pokémon".

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import os
import matplotlib.pyplot as plt

# --- 1. CONFIGURACIÓN ---
OUTPUT_DIR = "entrenamiento_pokemon_10000"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

DATA_PATH = "pokemon/pokemon_jpg/pokemon_jpg" 
BATCH_SIZE = 64 
noise_dim = 100
EPOCHS = 10000

## 2. Análisis y Preparación del Dataset
Se ha utilizado un dataset de 800 imágenes de Pokémon. El pipeline de datos incluye:
* **Redimensionamiento:** 64x64 píxeles.
* **Normalización:** Los valores de los píxeles se escalan al rango $[-1, 1]$.
* **Optimización:** Uso de `tf.data.Dataset.prefetch` para acelerar el entrenamiento.

In [ ]:
# Limpieza inicial
tf.keras.backend.clear_session()

# --- 2. DATASET ---
def preprocess(img):
    return (tf.cast(img, tf.float32) - 127.5) / 127.5

print("Cargando dataset...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH, label_mode=None, image_size=(64, 64), batch_size=BATCH_SIZE)
train_dataset = train_dataset.map(preprocess).shuffle(1000).prefetch(tf.data.AUTOTUNE)

## 3. Metodología: Diseño del Modelo
Se ha optado por una arquitectura **DCGAN** (Deep Convolutional GAN). Para asegurar la estabilidad tras detectar fallos iniciales, se aplicaron las siguientes técnicas:

1.  **TTUR (Two-Time-Scale Update Rule):** Se usa un ritmo de aprendizaje distinto. 
    * *Learning Rate* Generador: $1 \times 10^{-4}$
    * *Learning Rate* Discriminador: $2 \times 10^{-5}$
2.  **Label Smoothing:** Se etiquetan las imágenes reales como $0.9$ en lugar de $1.0$ para suavizar la pérdida.
3.  **Dropout:** Implementado en el Discriminador (0.3) para evitar que aprenda demasiado rápido y bloquee al Generador.

## 4. Funcionamiento Técnico del Algoritmo
El núcleo del proyecto es una **DCGAN**, que funciona mediante un juego de suma cero entre dos redes:

1. **El Generador:** Toma un vector de ruido latente $z \sim N(0, 1)$ y, mediante capas de convolución traspuesta (`Conv2DTranspose`), aprende a proyectar esos números aleatorios en una matriz de $64 \times 64 \times 3$ que represente a un Pokémon.
2. **El Discriminador:** Es un clasificador binario que intenta distinguir entre Pokémon reales del dataset y "falsificaciones" creadas por el generador.

### El Proceso de Aprendizaje
Durante cada paso de entrenamiento (`train_step`), ambas redes compiten:
* El Discriminador se entrena para maximizar la probabilidad de asignar la etiqueta correcta a las imágenes reales y falsas.
* El Generador se entrena para maximizar la probabilidad de que el Discriminador se equivoque, mejorando así su capacidad de crear imágenes realistas.

In [ ]:
# --- 3. MODELOS ---
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(noise_dim,)),
        layers.Dense(8*8*256, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((8, 8, 256)),
        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(3, 5, strides=2, padding="same", use_bias=False, activation="tanh")
    ])
    return model

def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(64, 64, 3)),
        layers.Conv2D(64, 5, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, 5, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [ ]:
generator = build_generator()
discriminator = build_discriminator()

# --- 4. OPTIMIZADORES Y PÉRDIDA ---
generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-5, beta_1=0.5)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

seed = tf.random.normal([16, noise_dim])

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([images.shape[0], noise_dim])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = cross_entropy(tf.ones_like(fake_output), fake_output)
        # Label smoothing para evitar que el discriminador sea demasiado agresivo
        real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)
        fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
        disc_loss = real_loss + fake_loss

    generator_optimizer.apply_gradients(zip(gen_tape.gradient(gen_loss, generator.trainable_variables), generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_tape.gradient(disc_loss, discriminator.trainable_variables), discriminator.trainable_variables))
    

## 5. Bitácora de Cambios y Resolución de Problemas

Durante el desarrollo, el proyecto pasó por varias fases críticas que obligaron a realizar cambios estructurales:

### A. El Problema del Mode Collapse (Colapso de Modo)
En otras versiones, el modelo sufría de colapso de modo severo. El generador descubría que ciertas imagenes (por ejemplo una mancha verdosa-rojiza) siempre lograba engañar al discriminador, por lo que dejó de producir variedad y empezó a repetir la misma imagen infinitamente.
* **Cambio realizado:** Se redujo el *Learning Rate* del Discriminador respecto al del Generador y se aumentó la tasa de `Dropout` al 0.3. Esto permitió que el Generador tuviera más "libertad" para explorar formas antes de que el Discriminador lo penalizara.

### B. Artefactos de Rejilla (Checkerboard Artifacts)
Se observaron patrones de cuadrícula en las imágenes generadas, un problema común en las capas `Conv2DTranspose`.
* **Cambio realizado:** Se ajustaron los tamaños de los filtros y los pasos para asegurar un solapamiento uniforme, y se aumentó el número de épocas para permitir que las capas de `BatchNormalization` estabilizaran las activaciones.

### C. Estabilidad en Entrenamiento Largo (10,000 Épocas)
Para evitar que el modelo "explotara" o se volviera inestable tras miles de iteraciones:
* Se implementó **Label Smoothing**: En lugar de usar etiquetas binarias perfectas (0 y 1), se usó 0.9 para las reales. Esto evita que el gradiente sea demasiado abrupto y mantiene el entrenamiento fluido durante las 10,000 épocas.

## 6. Interpretación de Resultados y Conclusiones
Tras un entrenamiento extensivo de **10,000 épocas**, se concluye:
* **Superación del Mode Collapse:** Al ajustar la velocidad del discriminador, el modelo logró salir del bucle de repetición y generar variedad.
* **Calidad Visual:** El modelo captura con éxito las paletas de colores y siluetas orgánicas características de la franquicia.
* **Mejoras Futuras:** Se recomienda aumentar la profundidad de filtros si se desea subir la resolución a 128x128.

In [ ]:
def save_and_plot(epoch):
    imgs = generator(seed, training=False)
    fig = plt.figure(figsize=(6,6))
    for i in range(16):
        plt.subplot(4, 4, i+1)
        plt.imshow((imgs[i] + 1) / 2)
        plt.axis('off')
    plt.savefig(f"{OUTPUT_DIR}/epoch_{epoch+1:05d}.png")
    plt.close()

In [ ]:
# --- 5. LOOP DE ENTRENAMIENTO ---
print(f"Iniciando entrenamiento épico de {EPOCHS} épocas...")

for epoch in range(EPOCHS):
    for image_batch in train_dataset:
        train_step(image_batch)
    
    current_epoch = epoch + 1
    
    # Lógica de visualización
    # 1. Los primeros 100 cada 20
    if current_epoch <= 100:
        if current_epoch % 20 == 0:
            print(f"Fase inicial: Época {current_epoch} guardada.")
            save_and_plot(epoch)
    
    # 2. A partir de 100 cada 100
    else:
        if current_epoch % 100 == 0:
            print(f"Fase larga: Época {current_epoch} guardada.")
            save_and_plot(epoch)
            
    Guardar el modelo cada 500 épocas (Backup de seguridad)
    if current_epoch % 500 == 0:
        generator.save(f"checkpoints/gen_checkpoint_{current_epoch}.h5")

# GUARDAR FINAL
generator.save("generador_pokemon_10000_final.h5")
print("Entrenamiento finalizado y modelo guardado")

## 7. Referencias y Fuentes
Para el desarrollo de este proyecto se han consultado las siguientes fuentes técnicas y conjuntos de datos:

* **Dataset Principal:** *Pokemon Images Dataset*, disponible en [Kaggle](https://www.kaggle.com/datasets/kvpratama/pokemon-images-dataset). Contiene las imágenes originales utilizadas para el entrenamiento del modelo.
* **Inspiración y Metodología:** Consulta de diversos proyectos de la comunidad de Kaggle sobre *Deep Convolutional Generative Adversarial Networks* (DCGANs) aplicadas a sprites de videojuegos y personajes de anime.
* **Documentación Técnica:**
    * TensorFlow Core: [Deep Convolutional Generative Adversarial Network tutorial](https://www.tensorflow.org/tutorials/generative/dcgan).
    * Ian J. Goodfellow et al. (2014): *Generative Adversarial Nets*.
    * Alec Radford et al. (2015): *Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks*.